# CySent Unsloth Training (Colab)
Fine-tune a CySent action model using a real dataset file named `cysent_action_dataset.jsonl`.

Before running this notebook, generate data locally:
`python scripts/build_cysent_dataset.py --rows 1500 --output cysent_action_dataset.jsonl`

Then upload `cysent_action_dataset.jsonl` to Colab (or place it in Google Drive).

In [2]:
# Clean conflicting preinstalled packages in Colab (safe to run repeatedly)
!pip uninstall -y unsloth unsloth-zoo unsloth_zoo trl peft datasets fsspec gcsfs bigframes > /dev/null 2>&1 || true

# Install required packages without strict version pinning for dependencies that unsloth may override.
# Let unsloth's installation from git manage the exact versions of its dependencies.
!pip install -q --upgrade pip
!pip install -q --no-cache-dir datasets peft trl fsspec
!pip install -q --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Verify the critical versions and fail fast if mismatched
import importlib.metadata as md
from packaging import version

v_unsloth = md.version('unsloth')
v_datasets = md.version('datasets')
v_trl = md.version('trl')
v_peft = md.version('peft')
v_fsspec = md.version('fsspec')

print('unsloth:', v_unsloth)
print('datasets:', v_datasets)
print('trl:', v_trl)
print('peft:', v_peft)
print('fsspec:', v_fsspec)

# Update assertions to reflect the versions currently installed by the latest unsloth from git.
# These versions were observed in the previous run's output.
assert v_datasets == '4.3.0'
assert v_trl == '0.24.0'
assert v_peft == '0.19.1'
assert v_fsspec == '2025.9.0'

# Optional: mount Google Drive if your dataset is there.
# from google.colab import drive
# drive.mount('/content/drive')

print('\nInstall check passed. If this is the first install in this runtime: Runtime > Restart runtime, then rerun Cell 1 once.')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
unsloth: 2026.4.6
datasets: 4.3.0
trl: 0.24.0
peft: 0.19.1
fsspec: 2025.9.0

Install check passed. If this is the first install in this runtime: Runtime > Restart runtime, then rerun Cell 1 once.


In [6]:
import os
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

# 1) Locate dataset (real dataset only)
dataset_candidates = [
    'cysent_action_dataset.jsonl',
    '/content/drive/MyDrive/cysent_action_dataset.jsonl',
    '/content/drive/MyDrive/CySent/cysent_action_dataset.jsonl',
]
dataset_path = next((p for p in dataset_candidates if os.path.exists(p)), None)
if dataset_path is None:
    raise FileNotFoundError(
        'Could not find cysent_action_dataset.jsonl. '
        'Upload it to Colab root or Google Drive, then rerun this cell.'
    )

# 2) Load and validate dataset size
ds = load_dataset('json', data_files=dataset_path, split='train')
print(f'Dataset path: {dataset_path}')
print(f'Total rows: {len(ds)}')
if len(ds) < 100:
    raise ValueError('Dataset too small for real training. Generate at least 100 rows (recommended: 1500+).')

split_ds = ds.train_test_split(test_size=0.05, seed=42)
train_ds = split_ds['train']
eval_ds = split_ds['test']
print(f'Train rows: {len(train_ds)} | Eval rows: {len(eval_ds)}')

# 3) Define valid actions and prompt format
VALID_ACTIONS = [
    'do_nothing', 'patch_hr_systems', 'patch_web_server', 'patch_auth_server',
    'rotate_credentials', 'isolate_suspicious_host', 'increase_monitoring', 'restore_backup',
    'deploy_honeypot', 'phishing_training', 'investigate_top_alert', 'segment_finance_database',
]
ACTION_SET = ', '.join(VALID_ACTIONS)

def format_chat(example):
    prompt = (
        f"You are CySent BLUE defender. Choose exactly one action from: {ACTION_SET}\n"
        f"Instruction: {example['instruction']}\n"
        f"State: {example['input']}\n"
        f"Action:"
    )
    return {'text': f"{prompt} {example['output']}"}

train_data = train_ds.map(format_chat, remove_columns=train_ds.column_names)
eval_data = eval_ds.map(format_chat, remove_columns=eval_ds.column_names)

# 4) Load base model with Unsloth
model_name = 'Qwen/Qwen2.5-3B-Instruct'
max_seq = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq,
    dtype=None,
    load_in_4bit=True,
)

# 5) Attach LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

# 6) Training args (real training profile)
use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
training_args = TrainingArguments(
    output_dir='outputs/cysent_checkpoint',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_steps=20,
    logging_steps=10,
    save_strategy='epoch',
    optim='adamw_8bit',
    lr_scheduler_type='linear',
    weight_decay=0.01,
    seed=42,
    report_to='none',
    bf16=use_bf16,
    fp16=not use_bf16,
)

# TRL API changed across versions; support both signatures.
try:
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_data,
        eval_dataset=eval_data,
        args=training_args,
        max_seq_length=max_seq,
        packing=False,
    )
except TypeError:
    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=train_data,
        eval_dataset=eval_data,
        args=training_args,
        max_seq_length=max_seq,
        packing=False,
    )

# 7) Train and save adapter
print('Starting training...')
trainer.train()

save_dir = 'outputs/cysent_unsloth_adapter'
os.makedirs(save_dir, exist_ok=True)
trainer.model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f'Adapter saved to {save_dir}')

Dataset path: /content/drive/MyDrive/cysent_action_dataset.jsonl
Total rows: 500
Train rows: 475 | Eval rows: 25
==((====))==  Unsloth 2026.4.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/475 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/25 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 475 | Num Epochs = 3 | Total steps = 90
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,3.413130
20,1.783831
30,0.290840
40,0.155593
50,0.141640
60,0.127394
70,0.120003
80,0.119790
90,0.110878


Adapter saved to outputs/cysent_unsloth_adapter


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import torch

# 1. Define action parser
def parse_action(text):
    lower = text.lower().strip()
    for action in VALID_ACTIONS:
        if action in lower:
            return action
    # Fallback to first word if it matches any action
    first_word = lower.split()[0] if lower.split() else ""
    for action in VALID_ACTIONS:
        if first_word == action:
            return action
    return 'do_nothing'

# 2. Test inference on trained model
FastLanguageModel.for_inference(model)  # Set to inference mode
test_state = "risk=0.72, attack=phishing_email, target=auth_server, compromised=1, credential_exposure=0.81"
test_prompt = (
    f"You are CySent BLUE defender. Choose exactly one action from: {ACTION_SET}\n"
    f"Instruction: Choose valid CySent action\n"
    f"State: {test_state}\n"
    f"Action:"
)

inputs = tokenizer(test_prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=20,
        temperature=0.0,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

raw_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
predicted_action = parse_action(raw_output.split("Action:")[-1])

print("=" * 60)
print("INFERENCE TEST")
print("=" * 60)
print(f"Test State: {test_state}")
print(f"Model Output: {raw_output.split('Action:')[-1].strip()}")
print(f"Parsed Action: {predicted_action}")
print("=" * 60)

Both `max_new_tokens` (=20) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/

INFERENCE TEST
Test State: risk=0.72, attack=phishing_email, target=auth_server, compromised=1, credential_exposure=0.81
Model Output: phishing_training
Explanation: Phishing training is the most appropriate defensive action in this scenario. The attack
Parsed Action: phishing_training


In [8]:
# 4. Validate saved adapter can be reloaded
print("\n✓ Training and inference complete!")
print("✓ Adapter saved. Validating reload...")

try:
    # Reload base model fresh with saved adapter
    model_reload, tokenizer_reload = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=max_seq,
        dtype=None,
        load_in_4bit=True,
    )
    from peft import PeftModel
    model_reload = PeftModel.from_pretrained(model_reload, "outputs/cysent_unsloth_adapter")
    FastLanguageModel.for_inference(model_reload)
    print("✓ Adapter reload successful!")
    print("✓ Ready for production use or HF Hub push.")
except Exception as e:
    print(f"⚠ Reload check failed: {e}")
    print("  But adapter files were saved. Check manually if needed.")


✓ Training and inference complete!
✓ Adapter saved. Validating reload...
==((====))==  Unsloth 2026.4.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
✓ Adapter reload successful!
✓ Ready for production use or HF Hub push.
